In [1]:
import geopandas as gpd
import pandas as pd

In [2]:
# spatial extent
agglom_extent_filepath = "../data/raw/zurich-extent.gpkg"

# AWS data (to get the hot-day periods)
aws_ts_df_filepath = "../data/interim/aws-ts-df.csv"

# UGZ data files
ugz_stations_filepath = (
    "../data/raw/ugz_stadtklima_zuerich_messorte_messnetz_meteoblue.csv"
)
ugz_ts_parquet_filepath = (
    "../data/raw/"
    "ugz_stadtklima_zuerich_temperaturmessungen_messnetz_meteoblue_2019-aktuell.parquet"
)
# AWEL data files (cached raw inputs, see `scripts/cache_awel_data.py`)
awel_ts_df_filepath = "../data/raw/awel-ts-df.csv"
awel_stations_filepath = "../data/raw/awel-stations.gpkg"
# output files
dst_ts_df_filepath = "../data/interim/lcs-ts-df.csv"
dst_stations_gdf_filepath = "../data/interim/lcs-stations.gpkg"

In [3]:
# Parameters
agglom_extent_filepath = "data/raw/zurich-extent.gpkg"
aws_ts_df_filepath = "data/interim/aws-ts-df.csv"
ugz_stations_filepath = "data/raw/ugz_stadtklima_zuerich_messorte_messnetz_meteoblue.csv"
ugz_ts_parquet_filepath = "data/raw/ugz_stadtklima_zuerich_temperaturmessungen_messnetz_meteoblue_2019-aktuell.parquet"
awel_ts_df_filepath = "data/raw/awel-ts-df.csv"
awel_stations_filepath = "data/raw/awel-stations.gpkg"
dst_ts_df_filepath = "data/interim/lcs-ts-df.csv"
dst_stations_gdf_filepath = "data/interim/lcs-stations.gpkg"


In [4]:
# load the hot-day periods identified in aws-data
aws_ts_df = pd.read_csv(
    aws_ts_df_filepath, index_col=["heatwave", "time"], parse_dates=["time"]
)
# get unique dates for filtering LCS data
hot_times = aws_ts_df.index.get_level_values("time")
hot_dates = hot_times.normalize().unique()

# also build a mapping from time -> heatwave label for reindexing LCS data
time_to_heatwave = (
    aws_ts_df.reset_index("heatwave")["heatwave"].groupby(level="time").first()
)
for year in sorted(hot_dates.year.unique()):
    year_dates = hot_dates[hot_dates.year == year]
    print(
        f"  {year}: {year_dates.min().date()} to {year_dates.max().date()} "
        f"({len(year_dates)} days)"
    )
print(f"Total: {len(hot_dates)} hot days")

  2022: 2022-06-15 to 2022-08-05 (20 days)
  2023: 2023-06-26 to 2023-08-24 (17 days)
  2024: 2024-06-19 to 2024-08-31 (19 days)
  2025: 2025-06-13 to 2025-08-16 (22 days)
Total: 78 hot days


In [5]:
# read UGZ parquet (covers 2019 to present)
ugz_long = pd.read_parquet(ugz_ts_parquet_filepath)

# deduplicate on (timestamp, locationID) and keep temperature only
ugz_long = (
    ugz_long[ugz_long["parameter"] == "T"]
    .drop_duplicates(subset=["timestamp", "locationID"])
    .copy()
)

# parse timestamps and filter to hot-day dates
ugz_long["timestamp"] = pd.to_datetime(ugz_long["timestamp"])
if ugz_long["timestamp"].dt.tz is not None:
    ugz_long["timestamp"] = ugz_long["timestamp"].dt.tz_convert(None)
ugz_long = ugz_long[ugz_long["timestamp"].dt.normalize().isin(hot_dates)]

# pivot to wide (time x location)
ugz_ts_df = ugz_long.pivot_table(
    index="timestamp", columns="locationID", values="value", aggfunc="mean"
)
ugz_ts_df.index.name = "time"
ugz_ts_df = ugz_ts_df.resample("h").mean()
ugz_ts_df.head()

locationID,A10,A7,B12,B3,B6,B9,C11,C14,C5,C8,...,R8,S16,S7,T18,Zürich-Affoltern,Zürich-Fluntern,Zürich-Kaserne,Zürich-Schimmelstrasse,Zürich-Wiedikon-Kollerwiese 1,Zürich-Wiedikon-Kollerwiese 2
time,,,,,,,,,,,,,,,,,,,,,
2022-06-15 00:00:00,NaN,NaN,NaN,17.650,15.225,NaN,15.200,16.250,16.725,17.375,...,14.050,15.000,12.900,15.550,12.400,15.525,18.875,19.550,NaN,NaN
2022-06-15 01:00:00,NaN,NaN,NaN,16.550,14.550,NaN,14.625,15.750,16.125,16.725,...,13.675,14.400,12.625,15.200,11.975,14.900,18.100,18.350,NaN,NaN
2022-06-15 02:00:00,NaN,NaN,NaN,15.875,14.875,NaN,14.900,15.700,16.200,16.375,...,13.825,14.700,13.125,15.475,12.100,14.975,17.975,17.800,NaN,NaN
2022-06-15 03:00:00,NaN,NaN,NaN,15.900,15.250,NaN,15.300,15.475,16.275,16.300,...,14.250,15.750,13.700,15.975,12.900,15.475,17.825,17.675,NaN,NaN
2022-06-15 04:00:00,NaN,NaN,NaN,15.725,15.925,NaN,15.650,16.125,16.125,16.400,...,15.450,16.475,14.825,16.250,13.550,15.675,17.900,18.050,NaN,NaN


In [6]:
# ACHTUNG: the AWEL series is read from a cached raw input rather than queried from the
# meteora client, which derives its station roster from the most recent monthly file
# only. Sensors that have since been relocated or decommissioned are therefore dropped
# from the whole historical record, and the client's month list also skips the first
# month whenever the requested period starts mid-month. Sensors that moved during the
# study period enter the cache as one station per location.
region = gpd.read_file(agglom_extent_filepath)["geometry"].iloc[:1]
awel_stations_gdf = (
    gpd.read_file(awel_stations_filepath).set_index("station_id").to_crs(region.crs)
)
# keep the sensors within the region, using the coordinates that they had while they
# were recording
awel_stations_gdf = awel_stations_gdf[awel_stations_gdf.within(region.iloc[0])]

awel_ts_df = pd.read_csv(awel_ts_df_filepath, index_col="time", parse_dates=["time"])
awel_ts_df = awel_ts_df[awel_ts_df.columns.intersection(awel_stations_gdf.index)]
awel_ts_df = awel_ts_df[awel_ts_df.index.normalize().isin(hot_dates)]
awel_ts_df = awel_ts_df.resample("h").mean()
awel_ts_df.head()

,2650,2651,2652,2653,2655,2657,2658,2659,2679,2680,...,2692,2695,2696,2697,2698,2807-1,2810,530,534,537
time,,,,,,,,,,,,,,,,,,,,,
2022-06-15 00:00:00,16.623333,15.981667,18.713333,14.740000,17.878333,16.516667,14.151667,18.423333,16.393333,20.106667,...,19.116667,18.048333,16.223333,15.606000,17.231667,13.598333,15.428000,15.965000,19.010000,15.046667
2022-06-15 01:00:00,16.065000,15.381667,18.052000,14.226000,17.208333,15.948333,13.766667,17.966667,16.103333,19.351667,...,18.515000,17.498333,15.738333,15.241667,16.704000,13.611667,14.884000,15.381667,18.311667,14.498000
2022-06-15 02:00:00,16.128333,15.191667,17.955000,14.658333,16.881667,15.881667,14.316667,17.526667,16.160000,18.692000,...,17.921667,17.053333,15.686667,15.011667,16.848333,14.096667,14.920000,15.423333,17.921667,14.850000
2022-06-15 03:00:00,16.238333,15.138333,17.675000,15.156667,16.588333,15.933333,14.543333,17.403333,16.061667,18.643333,...,17.721667,16.810000,15.598000,14.903333,16.893333,14.203333,15.068333,15.574000,17.596667,15.063333
2022-06-15 04:00:00,16.308333,15.176667,17.736667,15.953333,16.291667,15.824000,15.211667,17.618333,16.888333,18.522000,...,17.713333,16.863333,15.280000,15.046000,17.195000,15.158333,15.286667,15.872500,17.773333,15.358000


In [7]:
# combine UGZ and AWEL into a single time series
lcs_ts_df = pd.concat([ugz_ts_df, awel_ts_df], axis="columns")
# drop duplicate columns if any (e.g., overlapping stations)
lcs_ts_df = lcs_ts_df.loc[:, ~lcs_ts_df.columns.duplicated()]

# align to the heatwave MultiIndex from the AWS data
# map each hourly timestamp to its heatwave label
lcs_ts_df["heatwave"] = lcs_ts_df.index.map(time_to_heatwave)
lcs_ts_df = (
    lcs_ts_df.dropna(subset=["heatwave"]).reset_index().set_index(["heatwave", "time"])
)
lcs_ts_df.head()

A10  A7  B12      B3      B6  B9  \
heatwave              time                                                    
2022-06-15/2022-06-20 2022-06-15 00:00:00  NaN NaN  NaN  17.650  15.225 NaN   
                      2022-06-15 01:00:00  NaN NaN  NaN  16.550  14.550 NaN   
                      2022-06-15 02:00:00  NaN NaN  NaN  15.875  14.875 NaN   
                      2022-06-15 03:00:00  NaN NaN  NaN  15.900  15.250 NaN   
                      2022-06-15 04:00:00  NaN NaN  NaN  15.725  15.925 NaN   

                                              C11     C14      C5      C8  \
heatwave              time                                                  
2022-06-15/2022-06-20 2022-06-15 00:00:00  15.200  16.250  16.725  17.375   
                      2022-06-15 01:00:00  14.625  15.750  16.125  16.725   
                      2022-06-15 02:00:00  14.900  15.700  16.200  16.375   
                      2022-06-15 03:00:00  15.300  15.475  16.275  16.300   
                      2022-06-15 04:00:00  15.650  16.125  16.125  16.400   

                                           ...       2692       2695  \
heatwave              time                 ...                         
2022-06-15/2022-06-20 2022-06-15 00:00:00  ...  19.116667  18.048333   
                      2022-06-15 01:00:00  ...  18.515000  17.498333   
                      2022-06-15 02:00:00  ...  17.921667  17.053333   
                      2022-06-15 03:00:00  ...  17.721667  16.810000   
                      2022-06-15 04:00:00  ...  17.713333  16.863333   

                                                2696       2697       2698  \
heatwave              time                                                   
2022-06-15/2022-06-20 2022-06-15 00:00:00  16.223333  15.606000  17.231667   
                      2022-06-15 01:00:00  15.738333  15.241667  16.704000   
                      2022-06-15 02:00:00  15.686667  15.011667  16.848333   
                      2022-06-15 03:00:00  15.598000  14.903333  16.893333   
                      2022-06-15 04:00:00  15.280000  15.046000  17.195000   

                                              2807-1       2810        530  \
heatwave              time                                                   
2022-06-15/2022-06-20 2022-06-15 00:00:00  13.598333  15.428000  15.965000   
                      2022-06-15 01:00:00  13.611667  14.884000  15.381667   
                      2022-06-15 02:00:00  14.096667  14.920000  15.423333   
                      2022-06-15 03:00:00  14.203333  15.068333  15.574000   
                      2022-06-15 04:00:00  15.158333  15.286667  15.872500   

                                                 534        537  
heatwave              time                                       
2022-06-15/2022-06-20 2022-06-15 00:00:00  19.010000  15.046667  
                      2022-06-15 01:00:00  18.311667  14.498000  
                      2022-06-15 02:00:00  17.921667  14.850000  
                      2022-06-15 03:00:00  17.596667  15.063333  
                      2022-06-15 04:00:00  17.773333  15.358000  

[5 rows x 118 columns]

In [8]:
stations_crs = "epsg:2056"

# UGZ stations
ugz_stations_df = pd.read_csv(ugz_stations_filepath)
ugz_stations_gdf = gpd.GeoDataFrame(
    ugz_stations_df.set_index("locationID")[[]],
    geometry=gpd.points_from_xy(ugz_stations_df["EKoord"], ugz_stations_df["NKoord"]),
    crs=stations_crs,
)

# AWEL stations (index = station_id), already filtered to the region above
awel_stations_gdf = awel_stations_gdf[["geometry"]].to_crs(stations_crs)

# combine and keep only stations present in the time series
lcs_stations_gdf = pd.concat([ugz_stations_gdf, awel_stations_gdf]).rename_axis(
    "station_id"
)
lcs_stations_gdf = lcs_stations_gdf.loc[
    lcs_ts_df.columns.intersection(lcs_stations_gdf.index)
]
lcs_stations_gdf.head()

,geometry
A10,POINT (2678150 1248450)
A7,POINT (2677937 1250077)
B12,POINT (2678750 1247612)
B3,POINT (2678244 1252038)
B6,POINT (2678392 1250579)


In [9]:
# rename zurich kaserne (in ts_df and stations_gdf) to avoid conflicts with NABEL AWS
rename_dict = {"Zürich-Kaserne": "Zürich-Kaserne-UGZ"}
lcs_stations_gdf = lcs_stations_gdf.rename(index=rename_dict)
lcs_ts_df = lcs_ts_df.rename(columns=rename_dict)

In [10]:
lcs_ts_df.to_csv(dst_ts_df_filepath)
lcs_stations_gdf.reset_index().to_file(dst_stations_gdf_filepath)